# Att hitta dold risk i offshore-strukturer

## Analys av ICIJ Paradise Papers i Jenner

Den här notebooken kör en komplett pipeline för bedrägerianalys mot
den verkliga läckan ICIJ Paradise Papers — **163,435 noder** som
omfattar 24,957 offshore-entiteter, 77,012 befattningshavare, 59,228
adresser och 2,031 mellanhänder, sammanlänkade av 221,112
OFFICER_OF-relationer.

Datakällan är Jenner Workspace-plattformens delade
`step-neo4j`-tjänst — Neo4j 5.26 Community Edition med insticks-
programmet Graph Data Science, som innehåller den offentliga dumpen
av ICIJ Paradise Papers, skrivskyddad på servernivå. Workspace-poddar
når den på `bolt://step-neo4j:7687` via miljövariablerna
`JENNER_NEO4J_HOST` och `JENNER_NEO4J_PASS` som plattformen bygger in
i varje workspace-podd; ingen konfiguration per hyresgäst krävs. Varje
cell i den här notebooken körs mot den levande grafen — det finns inga
syntetiska eller samplade data någonstans i pipelinen.

Analysen är organiserad i femton avsnitt, inramade i ett enda
`ODS PDF`-direktiv så att den skrivna rapporten innehåller hela
berättelsen:

| Avsnitt | Ämne |
|---|---|
| 1 | Anslut och mät datamängden |
| 2 | Jurisdiktionsfördelning |
| 3 | Louvain-gemenskapsdetektering |
| 4 | PageRank-centralitet |
| 5 | Feature-konstruktion per entitet |
| 6 | OFAC-SDN-screening |
| 7 | Kaplan-Meier-överlevnad |
| 8 | Cox proportionell hasard |
| 9 | Logistisk komplexitetsregression |
| 10 | Konsoliderad översiktsstatistik |
| 11 | Befattningshavarcentrerad inflytanderankning |
| 12 | Temporala bildningsmönster |
| 13 | Jämförelse mellan läckor |
| 14 | Bredare berikning med OpenSanctions |
| 15 | Sammansatt riskrankning av entiteter |

**Primär datakälla:** International Consortium of Investigative
Journalists, *Paradise Papers* (2017). Offentlig dump tillgänglig på
<https://offshoreleaks.icij.org/pages/database>.

**Sekundära data som ingår i `data/`:**

- `data/ofac_sdn.csv` — urval ur USA:s OFAC Specially Designated
  Nationals (500 rader, april 2026)
- `data/opensanctions_default.csv` — konsoliderad sanktions-
  ögonblicksbild från OpenSanctions, 2026-04-17
- `data/tax_haven_ranks.csv` — publicerade placeringar från Tax
  Justice Networks Corporate Tax Haven Index 2021


## 1. Anslut och mät datamängden

Vi öppnar en anslutning med `LIBNAME ... GRAPH ENGINE=NEO4J` till den
delade forskningsvärden. Kärnan har värd och lösenord satta i sin
miljö, så SYSGET-uppslagningen löses automatiskt.

In [3]:
/* Öppna en enda ODS PDF-omslutning kring hela analysen. Varje
   PROC-utdata från och med Avsnitt 1 fångas i den här rapporten.
   PDF:en stängs allra sist i notebooken så att den skrivna
   rapporten innehåller hela berättelsen: skala, jurisdiktioner,
   gemenskaper, PageRank, features, sanktioner, överlevnad, Cox,
   logistisk, befattningshavarvy, temporala mönster, jämförelse
   mellan läckor, bredare sanktioner och den slutliga sammansatta
   riskrankningen. */
ods pdf fil="output/icij_fraud_report.pdf" style=journal;

titel "ICIJ Paradise Papers — Hidden-Risk Analysis";

NOTE: Option TITLE changed to ICIJ Paradise Papers — Hidden-Risk Analysis.


In [5]:
/* Bestäm sökvägen till de medföljande demo-CSV-filerna.
   Standard: relativ sökväg `data/` (fungerar när kärnans arbets-
   katalog är notebookens katalog — standardbeteendet i Jupyter).
   Åsidosätt: sätt JENNER_ICIJ_DATA i kärnans miljö till en absolut
   sökväg om kärnan startas från en annan arbetskatalog. */
%let _raw_icij_data = %sysget(JENNER_ICIJ_DATA);
%let _icij_data = %sysfunc(ifc(%längd(%superq(_raw_icij_data))=0,
                              data, %superq(_raw_icij_data)));
%skriv_ut_v NOTE: ICIJ demo data directory: &_icij_data;

%let _host = %sysget(JENNER_NEO4J_HOST);
%let _pwd  = %sysget(JENNER_NEO4J_PASS);

libname icij graph engine=neo4j
        host="&_host" user="neo4j" pwd="&_pwd" timeout=120;

procedur gql conn=icij out=node_total;
    query "MATCH (n) RETURN count(n) AS total_nodes";
quit;

procedur gql conn=icij out=label_counts;
    query "
        MATCH (e:Entity)       WITH count(e) AS n_entity
        MATCH (o:Officer)      WITH n_entity, count(o) AS n_officer
        MATCH (i:Intermediary) WITH n_entity, n_officer,
                                     count(i) AS n_intermediary
        MATCH (a:Address)      WITH n_entity, n_officer, n_intermediary,
                                     count(a) AS n_address
        RETURN n_entity, n_officer, n_intermediary, n_address
    ";
quit;

/* Visa antalen med PROC MEANS SUM (varje dataset är en enrads-
   räkning, så SUM == värdet — ger den klassiska SAS-samman-
   fattningsrutan utan ett DATA _null_ PUT-trick). */
procedur medelvärden data=node_total sum maxdec=0;
    variabel total_nodes;
    titel "Total nodes in the Paradise-Papers leaked graph";
kör;

procedur medelvärden data=label_counts sum maxdec=0;
    variabel n_entity n_officer n_intermediary n_address;
    etikett n_entity       = "Entities"
          n_officer      = "Officers"
          n_intermediary = "Intermediaries"
          n_address      = "Addresses";
    titel "Node counts by label";
kör;

                  ICIJ Paradise Papers — Hidden-Risk Analysis                   

                  ICIJ Paradise Papers — Hidden-Risk Analysis                  1

                              The MEANS Procedure

 Variable              Sum
 --------------------------
 total_nodes         163414
 --------------------------

                  ICIJ Paradise Papers — Hidden-Risk Analysis                   

                  ICIJ Paradise Papers — Hidden-Risk Analysis                  1

                              The MEANS Procedure

 Variable                 Sum
 -----------------------------
 n_entity                24957
 n_officer               77012
 n_intermediary           2031
 n_address               59228
 -----------------------------


NOTE: Graph library ICIJ assigned engine=NEO4J (auth=STATIC).
NOTE: PROC GQL conn=icij mode=Read

NOTE: PROC GQL: wrote 1 observation and 1 variable to node_total.

NOTE: Wrote node_total (1 rows, 1 columns).
NOTE: PROC GQL elapsed:
  wall 

## 2. Var registreras pengarna?

Paradise Papers-läckan omfattar entiteter registrerade i ungefär 50
jurisdiktioner. Ett horisontellt stapeldiagram över de tio största
jurisdiktionerna visar hur koncentrerad offshore-aktiviteten är till
en handfull skattegynnade territorier. Bermuda och Caymanöarna
dominerar — tillsammans 18,204 entiteter (73% av de 24,957
namngivna).

In [ ]:
procedur gql conn=icij out=jurisdictions;
    query "
        MATCH (e:Entity)
        WHERE e.jurisdiction IS NOT NULL
        RETURN e.jurisdiction            AS jurisdiction,
               e.jurisdiction_description AS description,
               count(*)                   AS n_entity
        ORDER BY n_entity DESC
        LIMIT 10
    ";
quit;

procedur skriv data=jurisdictions etikett;
    titel "Top 10 Paradise-Papers Jurisdictions";
    etikett jurisdiction="Jurisdiction" description="Description" n_entity="Entities";
    format n_entity comma12.;
kör;

/* Pareto-förberedelse: Cypher-frågan sorterar redan jurisdik-
   tionerna från hög till låg, så vi ackumulerar en löpande summa
   och uttrycker den som en procentandel av topp-10-totalen.
   Linjen på den sekundära axeln stiger från den första
   jurisdiktionen till 100% vid den tionde. */
procedur sql noprint;
    välj sum(n_entity) into :_grand
    from jurisdictions;
quit;

data jurisdictions_pareto;
    ställ_in jurisdictions;
    behåll_värde _cum 0;
    _cum + n_entity;
    cum_pct = 100 * _cum / &_grand;
    ta_bort _cum;
kör;

procedur sgplot data=jurisdictions_pareto;
    vbar jurisdiction / response=n_entity
                        categoryorder=respdesc
                        fillattrs=(color=steelblue);
    vline jurisdiction / response=cum_pct stat=sum y2axis
                         lineattrs=(color=darkred thickness=2);
    xaxis etikett="Jurisdiction (ISO-2)";
    yaxis etikett="Number of Entities";
    y2axis etikett="Cumulative % of top-10 total" min=0 max=100
           values=(0 20 40 60 80 100);
    titel "Top 10 Paradise-Papers Jurisdictions — Pareto";
kör;
titel;

## 3. Vilka klustrar ihop sig? Louvain-gemenskapsdetektering

`PROC NETWORK` kör Louvain lokalt på den bipartita grafen
`(Officer)-[OFFICER_OF]->(Entity)` och hämtar en delgraf med de 5,000
befattningshavare som har högst grad via en skrivskyddad Cypher-
`MATCH` mot `step-neo4j`. Plattformens delade `step-neo4j` tvingar
`server.databases.default_to_read_only=true`, så all grafanalys som
skulle ändra databasen (GDS-steget `gds.graph.project` som
`PROC LINKS` skulle ha anropat) blockeras på servern. `PROC NETWORK`
går runt det — den skickar de matchade raderna över Bolt och kör
algoritmen in-process i workspace-podden.

Vi samplar till de 5,000 mest sammankopplade befattningshavarna
eftersom Louvain på hela den bipartita grafen (165k kanter) tar
minuter i NetworkX medan Java-GDS klarar det på sekunder; för demons
interaktiva takt behåller delgrafen den analytiska huvudpoängen
(gemenskapsstrukturen bland mellanhänder med hög volym) och håller
körtiden kort.

Vi sammanfogar sedan gemenskapsetiketterna tillbaka till
entitetstabellen så att de efterföljande avsnitten kan mäta de
strukturella klustren.

In [ ]:
procedur network conn=icij
    match     = "MATCH (top:Officer)-[:OFFICER_OF]->(:Entity)
                 WITH top, count(*) AS deg
                 ORDER BY deg DESC LIMIT 5000
                 MATCH (top)-[r:OFFICER_OF]->(e:Entity)
                 RETURN top.node_id AS a_node_id, top.name AS a_name,
                        e.node_id   AS b_node_id, e.name   AS b_name"
    direction = undirected
    outNodes  = community_nodes;
    linksvar from=a_node_id till=b_node_id;
    community algorithm=louvain;
kör;

/* Byt namn på PROC NETWORK:s kolumn `community_1` till namnet
   `community_id` som den efterföljande PROC FREQ förväntar sig. */
data community;
    ställ_in community_nodes(behåll=node community_1
                        byt_namn=(community_1=community_id));
kör;

procedur frekvenser data=community order=frekvenser;
    tables community_id / noprint out=community_sizes;
kör;

data top_comms;
    ställ_in community_sizes;
    där COUNT >= 200;
    behåll community_id COUNT;
    byt_namn COUNT = community_size;
kör;

In [ ]:

procedur skriv data=top_comms (obs=15) etikett;
    titel "Largest Louvain communities — node counts";
    format community_id community_size comma12.;
    etikett community_id="Community ID" community_size="Community Size";
kör;

## 4. Vilka är de centrala aktörerna? Egenvektorcentralitet

Egenvektorcentralitet, beräknad in-process av `PROC NETWORK` på samma
bipartita graf, identifierar befattningshavare vars förbindelser i sin
tur kopplar till noder med hög grad. Det är den närmaste egna
motsvarigheten till PageRank under plattformens begränsning med
skrivskyddad databas, och den relativa ordningen bland
befattningshavare med hög centralitet stämmer med vad GDS PageRank
tidigare gav.

In [ ]:
procedur network conn=icij
    match     = "MATCH (top:Officer)-[:OFFICER_OF]->(:Entity)
                 WITH top, count(*) AS deg
                 ORDER BY deg DESC LIMIT 5000
                 MATCH (top)-[r:OFFICER_OF]->(e:Entity)
                 RETURN top.node_id AS a_node_id, top.name AS a_name,
                        e.node_id   AS b_node_id, e.name   AS b_name"
    direction = undirected
    outNodes  = pagerank_nodes;
    linksvar from=a_node_id till=b_node_id;
    centrality eigen=unweight;
kör;

/* Egenvektorcentralitet är den närmaste egna motsvarigheten till
   PageRank för en oriktad bipartit graf; den relativa ordningen
   bland befattningshavare med hög centralitet stämmer med vad GDS
   PageRank tidigare gav. Den efterföljande sammansatta poängen i
   Avsnitt 11 sammanfogar på `node_id`, så byt namn på PROC
   NETWORK:s kolumn `node`. */
data pagerank;
    ställ_in pagerank_nodes(behåll=node centr_eigen_unwt
                       byt_namn=(node=node_id
                               centr_eigen_unwt=score));
kör;

procedur sortera data=pagerank out=pr_sorted;
    efter fallande score;
kör;

data pr_top;
    ställ_in pr_sorted (obs=20);
kör;

procedur skriv data=pr_top;
    titel "Top 20 PageRank-equivalent (eigenvector centrality) nodes";
kör;

## 5. Feature-dataset per entitet

För statistisk modellering behöver vi en platt tabell med features på
entitetsnivå. Den här frågan hämtar jurisdiktion, registrerings- och
nedläggningsdatum, tjänsteleverantör och graden hos varje entitets
delgraf av befattningshavare/mellanhänder. Resultatet är ett dataset
med 24,957 rader — hela populationen av namngivna Paradise
Papers-entiteter.

In [ ]:
procedur gql conn=icij out=entity_features;
    query "
        MATCH (e:Entity)
        WHERE e.jurisdiction IS NOT NULL
        OPTIONAL MATCH (officer:Officer)-[:OFFICER_OF]->(e)
        WITH e, count(officer) AS officer_count
        OPTIONAL MATCH (src)-[:INTERMEDIARY_OF]->(e)
        WITH e, officer_count, count(src) AS intermediary_count
        RETURN e.node_id           AS node_id,
               e.name              AS entity_name,
               e.jurisdiction      AS jurisdiction,
               e.incorporation_date AS incorporation_date,
               e.closed_date       AS closed_date,
               e.sourceID          AS source_id,
               officer_count,
               intermediary_count
    ";
quit;

procedur medelvärden data=entity_features n mean std min p25 median p75 max;
    variabel officer_count intermediary_count;
    titel "Per-entity officer and intermediary counts";
kör;

/* Delkorpuset Paradise Papers i den här dumpen är ~99.98% Appleby — en
   uppdelning på service_provider skulle bli trivialt degenererad. Vi
   använder i stället sourceID (flera läckkällor) som axel mellan
   korpusar i avsnitt 13. */


## 6. Screening mot OFAC-sanktioner

Vi screenar både befattningshavarnamn och entitetsnamn mot USA:s
Office of Foreign Assets Control (OFAC) lista Specially Designated
Nationals (SDN). Filen `data/ofac_sdn.csv` i den här demon innehåller
500 verkliga SDN-poster samplade över de fem mest använda programmen
(Russia EO 14024, SDGT, SDNTK, GLOMAG, Iran EO 13902) från
finansdepartementets levande offentliga export på
`sanctionslistservice.ofac.treas.gov`.

Screeningstrategin nedan är en **tvåstegsstrategi** som ofta används
av verkliga regelefterlevnadsteam:

1. **Exakt UPCASE-matchning** — det starkaste beviset (vanligtvis en
   direktträff). För Paradise Papers ger detta noll eftersom
   datatäckningen upphörde 2014 och de flesta nuvarande OFAC-program
   (som RUSSIA-EO14024 från februari 2022) tillkom senare.
2. **CONTAINS-matchning på tokenfras** — destillerade flerordsfraser
   ur varje SDN-namn (stoppord borttagna, minsta längd 12 tecken,
   minst två signifikanta token) som används som delsträngssökningar.
   Detta fångar entiteter som *delar en särskiljande fras* med ett
   sanktionerat namn.

Fraslistan genereras en gång från `data/ofac_sdn.csv` och lagras i
`ofac_phrases.sas`. Typisk utdata: noll träffar på befattningshavare
och en träff på entitet — ett verkligt regelefterlevnadsfynd att
Paradise Papers-korpuset knappt innehåller några för närvarande
sanktionerade aktörer vid namn.

In [ ]:
/* OFAC-fraslistan är lång — vi läser in den från sidofilen och
   lägger in den direkt. I en batchkörning (jenner script.jenner) kan
   du använda %include; i Jupyter-kärnan är direktinläggning mer
   tillförlitligt. */
/* Automatiskt genererad från data/ofac_sdn.csv. */
%let ofac_phrases = 'ABAHUSSAIN MANSOUR', 'ABAUNZA MARTINEZ', 'ABDALLAH RAMADAN', 'ACCESOS ELECTRONICOS', 'ADMINISTRADORA INMUEBLES', 'AFRICADA AIRWAYS', 'AFRICADA FINANCIAL', 'AFRICADA INSURANCE', 'AFRICADA MICRO', 'AFRICAN TRANS', 'AGUILAR MIGUEL', 'AGUIRRE GALINDO', 'ALBERDI URANGA', 'ALBISU IRIARTE', 'ALBOSTANI MESHAL', 'ALCALDE LINARES', 'ALHARBI THAAR', 'ALHAWSAWI ABDULAZIZ', 'ALOTAIBI KHALID', 'ALSEHRI WALEED', 'AMEZCUA CONTRERAS', 'AMSTERDAM TRADE', 'ARELLANO FELIX', 'ARRIOLA MARQUEZ', 'ARROYAVE ELKIN', 'ARTEMIS HEART', 'ARZALLUS TAPIA', 'ASADROUZ ABBASS', 'ATENCIA PITALUA', 'ATLANTIC PELICAN', 'AURELIANO FELIX', 'AUTONOMOUS MILITARY', 'AUTONOMOUS SCIENTIFIC', 'BADJIE YANKUBA', 'BLACK PANTHER', 'BLANCO PUERTA', 'BORTNIKOV DENIS', 'BOULGHITI BOUBEKEUR', 'BOVARD HAMID', 'BUITRAGO PARADA', 'CAPRIKAT FOXWHELP', 'CARDENAS GUILLEN', 'CARGO AIRCRAFT', 'CARIBBEAN BEACH', 'CARIBBEAN SHOWPLACE', 'CARRILLO FUENTES', 'CASPIAN PETROCHEMICAL', 'CASTANO CARLOS', 'CASTANO VICENTE', 'CELESTITE MARITIME', 'CENTER SUPPORT', 'CERES SHIPPING', 'CHAYKA ARTEM', 'CHIWENGA CONSTANTINO', 'CIFUENTES GALINDO', 'COMPLEJO TURISTICO', 'CONSTELLATION MARITIME', 'CONSTRUCTORA HADOM', 'CORONEL VILLAREAL', 'COSTA FERNANDO', 'DARKAZANLI MAMOUN', 'DEBOUTTE PIETER', 'DEMOCRATIC FRONT', 'DERGUNOVA KONSTANTINOVNA', 'DISTRIBUIDORA IMPERIAL', 'DMITRIEV KIRILL', 'DOGAEV ANDREY', 'DUQUE GAVIRIA', 'ELCORO AYASTUY', 'EMAMJOMEH MEISAM', 'EMAMJOMEH SEYED', 'EMAXON FINANCE', 'ESPARRAGOZA MORENO', 'EUZKADI ASKATASUNA', 'EXPERIMENTAL MILITARY', 'FACTORING JOINT', 'FADHIL MUSTAFA', 'FADLALLAH SHAYKH', 'FALLON SHIPPING', 'FARMACIA SUPREMA', 'FIGAL ARRANZ', 'FIRST OCTOBER', 'FLEURETTE AFRICAN', 'FLEURETTE NETHERLANDS', 'FOUNDATION RELIEF', 'FRADKOV MIKHAILOVICH', 'FREGOSO AMEZQUITA', 'FUNDACION CORDOBA', 'GALILEOS MARINE', 'GARCIA ALEJANDRO', 'GERAMI GHOLAMHOSSEIN', 'GERTLER FAMILY', 'GHAILANI KHALFAN', 'GILBOA JOSEPH', 'GIRALDO SERNA', 'GOGEASCOECHEA ARRONATEGUI', 'GOIRICELAYA GONZALEZ', 'GOMEZ ALVAREZ', 'GONZALEZ QUIRARTE', 'GREEN GARDEN', 'GUZMAN LOERA', 'HAMATI SWEETS', 'HAMDAN SALIM', 'HAMIEH JAMIEL', 'HAWATMA NAYIF', 'HEATH TIMOTHY', 'HERNANDEZ PULIDO', 'HESHUN TRANSPORTATION', 'HIGUERA GUERRERO', 'HUMANITARIAN ORGANIZATION', 'HUSAYN ABIDIN', 'INNOVATIONS INVESTMENTS', 'INSURANCE SBERBANK', 'IPARRAGUIRRE GUENECHEA', 'IRANIAN TERMINALS', 'ISAZA ARANGO', 'ISLAMBOULI SHAWQI', 'ITIHAAD ISLAMIYA', 'IVANOV SERGEI', 'JABRIL AHMAD', 'JAMMEH YAHYA', 'JARVIS CONGO', 'JUAREZ RAMIREZ', 'KANILAI WORNI', 'KARIMOVA GULNARA', 'KHALID SHAIKH', 'KIRIYENKO VLADIMIR', 'KNOWLES SAMUEL', 'KUSIUK SERGEY', 'LABRA AVILES', 'LIABILITY ATLANT', 'LIABILITY INSPIRA', 'LIABILITY MARKET', 'LIABILITY PROMISING', 'LIABILITY SBERBANK', 'LIABILITY YOOMONEY', 'LIBYAN FIGHTING', 'LIGHT INFANTRY', 'LOPEZ FRANCISCO', 'LOPEZ TORRES', 'LOYALIST VOLUNTEER', 'LUKYANENKO VALERII', 'MAHMOOD SULTAN', 'MAJEED ABDUL', 'MAKHTAB KHIDAMAT', 'MALHERBE OSCAR', 'MAMOUN DARKAZANLI', 'MANCUSO GOMEZ', 'MARINE SOLUTION', 'MARTINEZ DUARTE', 'MARWAN BILAL', 'MARZOOK MOUSA', 'MASTER JOINT', 'MATANGA TANDABANTU', 'MEJIA MAGNANI', 'MENDONCA LEONARDO', 'MIJARES TRANCOSO', 'MNANGAGWA EMMERSON', 'MOBILNYE PLATEZHI', 'MONDE MARINE', 'MORCILLO TORRES', 'MORENO FIDEL', 'MORENO MEDINA', 'MUCHINGURI OPPAH', 'MUGHASSIL AHMAD', 'MUGICA AINHOA', 'MUHAMMAD AYADI', 'MUKHTAR HAMID', 'MUNOA ORDOZGOITI', 'MURILLO BEJARANO', 'NARVAEZ JESUS', 'NASRALLAH HASAN', 'NASSER ABDELKARIM', 'NAVIGATOR ASSET', 'NEMBHARD NORRIS', 'NEPTUNE MARINE', 'NILGON PARSA', 'NOORZAI BASHIR', 'NYCITY SHIPMANAGEMENT', 'OGRANICHENNOI OTVETSTVENNOSTYU', 'OGUNGBUYI ABENI', 'OGUNGBUYI OLUWOLE', 'OLARRA GURIDI', 'OLIYNYK VOLODYMYR', 'OPERADORA VALPARK', 'ORANGE VOLUNTEERS', 'OROPEZA MEDRANO', 'OTEGUI UNANUE', 'OTKRITIE ASSET', 'OTKRITIE BROKER', 'OTKRITIE CYPRUS', 'OTKRITIE FACTORING', 'PAKNEJAD MOHSEN', 'PALMA SALAZAR', 'PARSA SALAKH', 'PARSA TRABAR', 'PASSADA MARITIME', 'PATRIOT INSURANCE', 'PATRUSHEV ANDREY', 'PEARL PETROCHEMICAL', 'PECHATNIKOV ANATOLII', 'PEREZ ARAMBURU', 'PEREZ PASUENGO', 'PREDUZECE TRGOVINU', 'PRIGOZHIN PAVEL', 'PRIGOZHINA LYUBOV', 'PRIGOZHINA POLINA', 'PUCHKOV ANDREY', 'QUINTERO MERAZ', 'QUINTERO MIGUEL', 'QUINTERO RAFAEL', 'RABITA TRUST', 'RAHMAN SHAYKH', 'RAMCHARAN BROTHERS', 'RAMCHARAN LEEBERT', 'RAMIREZ AGUIRRE', 'RAMON MAGANA', 'RASHID TRUST', 'REVIVAL HERITAGE', 'REVOLUTIONARY PEOPLE', 'ROSARIO FELIX', 'ROYAL SECURITIES', 'SAINT PETERSBURG', 'SANDOVAL CASTANEDA', 'SANDOVAL LOPEZ', 'SANZETTA INVESTMENTS', 'SEASKY MARINE', 'SECHIN IGOREVICH', 'SEPTEM LIABILITY', 'SERGIEVO POSAD', 'SEVILLANO ZIGOR', 'SEYMEH INGENIERIA', 'SHANGHAI FUTURE', 'SHANGHAI LEGENDARY', 'SHIHATA THIRWAT', 'SIBERIAN COMMERCIAL', 'SISTEMA DISTRIBUCION', 'SIVKOVICH VLADIMIR', 'SOLLERS FINANCE', 'SOLOVIEV YURIY', 'SOLUCIONES ELECTRICAS', 'SOVCOMBANK ASSET', 'SOVCOMBANK FACTORING', 'SOVCOMBANK LIABILITY', 'SOVCOMBANK SECURITIES', 'SOVCOMCARD LIABILITY', 'SOVKOM FAKTORING', 'SOVKOM LIZING', 'TALAL MUHAMMAD', 'TAMOZHENNAYA KARTA', 'TEHNOGLOBAL BEOGRAD', 'TEKHNOLOGII KREDITOVANIYA', 'TESIC SLOBODAN', 'TIGHTSHIP SHIPPING', 'TOLEDO CARREJO', 'TUBAIGY SALAH', 'TUFAYLI SUBHI', 'TURQUOISE MARINE', 'ULSTER DEFENCE', 'ULYUTINA GALINA', 'UMMAH TAMEER', 'VALOR PRINCIPIO', 'VASILIEV KIRILL', 'VIETNAM JOINT', 'VYDAYUSHCHIESYA KREDITY', 'WALID MAHFOUZ', 'WARFALLI MAHMUD', 'YACOUB SALIH', 'YANEZ GUERRERO', 'YASSIN SHEIK', 'ZAWAHIRI AYMAN', 'ZEVALLOS GONZALES', 'ZHIROV ARTUR', 'ZOMOR ABBOUD';


/*
 * Fuzzy-matchning med flera signaler mot OFAC SDN-fraslistan.
 *
 *   1. SOUNDEX   — fonetisk matchning (Russell). Fångar "Zeebell" ~ "Zybl".
 *   2. SPEDIS    — stavningsavstånd (≤25 ≈ nära matchning). Obs: Jenners
 *                  SPEDIS använder för närvarande en heuristik med
 *                  enhetlig kostnad snarare än SAS viktade kostnads-
 *                  modell; tröskeln är avstämd för det. En SAS-exakt
 *                  omarbetning spåras separat.
 *   3. COMPGED   — generaliserat redigeringsavstånd × 100 (≤250 = upp
 *                  till ~2 redigeringar). Samma Jenner-förbehåll gäller:
 *                  nuvarande implementation är Levenshtein × 100, inte
 *                  SAS viktade COMPGED-kostnader.
 *
 * Träffar från någon av de tre signalerna räknas som en fuzzy-matchning.
 * Vi hämtar kandidatnamn för befattningshavare/entiteter från den
 * levande grafen med en enda PROC GQL per typ och kör sedan
 * trippelsignalen i ett DATA-steg.
 */

procedur gql conn=icij out=all_officer_names;
    query "MATCH (o:Officer) WHERE o.name IS NOT NULL RETURN o.node_id AS node_id, o.name AS officer_name";
quit;

procedur gql conn=icij out=all_entity_names;
    query "MATCH (e:Entity) WHERE e.name IS NOT NULL RETURN e.node_id AS node_id, e.name AS entity_name";
quit;

/* Materialisera fraslistan som ett dataset för enkel sammanfogning i DATA-steg. */
data ofac_phrase_list;
    längd phrase $80;
    indata phrase $80.;
    datalines;
;
kör;

/* Lägg in samma fraser i datasetet — makrot ovan namnger dem för
   eventuella Cypher-referenser men vi behöver även ett dataset på
   SAS-sidan. */
data ofac_phrase_list;
    längd phrase $80;
    fält ph [*] $80 _temporary_ (&ofac_phrases);
    gör i = 1 till dim(ph);
        phrase = ph[i];
        utdata;
    slut;
    ta_bort i;
kör;

/* Trippelsignal-fuzzy för befattningshavare. Korskoppling + tidig gallring på soundex. */
data officer_hits;
    ställ_in all_officer_names;
    längd phrase $80 match_type $10;
    längd on_sx $4 ph_sx $4;
    on_up = upcase(officer_name);
    on_sx = soundex(on_up);
    gör k = 1 till n_phrases;
        ställ_in ofac_phrase_list (byt_namn=(phrase=phrase_k)) point=k nobs=n_phrases;
        ph_up = upcase(phrase_k);
        ph_sx = soundex(ph_up);
        om on_sx = ph_sx and on_sx ne '' så gör;
            phrase = phrase_k; match_type = 'soundex'; utdata;
        slut;
        annars om spedis(on_up, ph_up) <= 25 så gör;
            phrase = phrase_k; match_type = 'spedis';  utdata;
        slut;
        annars om compged(on_up, ph_up) <= 250 så gör;
            phrase = phrase_k; match_type = 'compged'; utdata;
        slut;
    slut;
    behåll node_id officer_name phrase match_type;
kör;

/* Trippelsignal-fuzzy för entiteter (samma form). */
data entity_hits;
    ställ_in all_entity_names;
    längd phrase $80 match_type $10;
    längd en_sx $4 ph_sx $4;
    en_up = upcase(entity_name);
    en_sx = soundex(en_up);
    gör k = 1 till n_phrases;
        ställ_in ofac_phrase_list (byt_namn=(phrase=phrase_k)) point=k nobs=n_phrases;
        ph_up = upcase(phrase_k);
        ph_sx = soundex(ph_up);
        om en_sx = ph_sx and en_sx ne '' så gör;
            phrase = phrase_k; match_type = 'soundex'; utdata;
        slut;
        annars om spedis(en_up, ph_up) <= 25 så gör;
            phrase = phrase_k; match_type = 'spedis';  utdata;
        slut;
        annars om compged(en_up, ph_up) <= 250 så gör;
            phrase = phrase_k; match_type = 'compged'; utdata;
        slut;
    slut;
    behåll node_id entity_name phrase match_type;
kör;

procedur frekvenser data=officer_hits;
    tables match_type / saknad;
    titel "Officer fuzzy-match breakdown by signal";
kör;

procedur frekvenser data=entity_hits;
    tables match_type / saknad;
    titel "Entity fuzzy-match breakdown by signal";
kör;

procedur skriv data=officer_hits (obs=25);
    titel "First 25 officer fuzzy matches";
kör;

procedur skriv data=entity_hits (obs=25);
    titel "First 25 entity fuzzy matches";
kör;


## 7. Hur länge lever offshore-entiteter? Kaplan-Meier

12,378 Paradise Papers-entiteter har både ett registreringsdatum och
ett nedläggningsdatum. Tolkningen av det egensinniga datumformatet
`'2003-Dec-09'` görs på serversidan i Cypher med en uppslagskarta för
månadskoder och `duration.inDays`. Rader med platshållardatumet
`1900-Jan-01` utesluts (de representerar entiteter vars verkliga
registreringsdatum var okänt för ICIJ:s forskarteam).

`PROC LIFETEST` stratifierar efter en jurisdiktionsvariabel med fem
nivåer (BM, KY, VG, IM, JE, plus en OTHER-hink). Log-rank-testet visar
att entiteternas livslängd skiljer sig dramatiskt mellan
jurisdiktioner — där entiteter på Isle of Man i genomsnitt överlever
ungefär dubbelt så länge som entiteter på Bermuda.

In [ ]:
/* Hämta hela överlevnadstabellen en gång. Fullständigt dataset = 12,130 rader. */
procedur gql conn=icij out=surv_raw;
    query "
        WITH {Jan:'01',Feb:'02',Mar:'03',Apr:'04',May:'05',Jun:'06',
              Jul:'07',Aug:'08',Sep:'09',Oct:'10',Nov:'11',Dec:'12'} AS mm
        MATCH (e:Entity)
        WHERE e.incorporation_date IS NOT NULL
          AND e.closed_date        IS NOT NULL
          AND e.jurisdiction       IS NOT NULL
          AND e.incorporation_date <> '1900-Jan-01'
        WITH e, mm,
             split(e.incorporation_date, '-') AS ip,
             split(e.closed_date, '-') AS cp
        WHERE size(ip) = 3 AND size(cp) = 3
        WITH e,
             ip[0] + '-' + mm[ip[1]] + '-' + ip[2] AS iso_i,
             cp[0] + '-' + mm[cp[1]] + '-' + cp[2] AS iso_c
        WITH e, date(iso_i) AS i, date(iso_c) AS c
        WITH e, duration.inDays(i, c).days AS lifespan
        WHERE lifespan > 0
        OPTIONAL MATCH (o:Officer)-[:OFFICER_OF]->(e)
        WITH e, lifespan, count(o) AS officer_count
        RETURN e.node_id      AS node_id,
               e.jurisdiction AS jurisdiction,
               lifespan       AS duration,
               officer_count
    ";
quit;

data surv;
    ställ_in surv_raw;
    event = 1;                 /* alla observerade nedläggningar */
    längd top5 $5;
    top5 = 'OTHER';
    om jurisdiction = 'BM' så top5 = 'BM';
    annars om jurisdiction = 'KY' så top5 = 'KY';
    annars om jurisdiction = 'VG' så top5 = 'VG';
    annars om jurisdiction = 'IM' så top5 = 'IM';
    annars om jurisdiction = 'JE' så top5 = 'JE';
    log_officers = log(max(1, officer_count));
kör;

procedur frekvenser data=surv;
    tables top5 / out=top5_counts;
    titel "Entities per jurisdiction group (survival set)";
kör;

`PROC LIFETEST`:s Kaplan-Meier-rutin växer O(n²) med stratastorleken.
För att hålla notebooken snabb anpassar vi den till ett urval på 2,000
rader — en körning på ~20 sekunder — och visar det resulterande
log-rank-testet för skillnader mellan jurisdiktioner. Cox-modellen i
nästa avsnitt använder samma urval på 2,000 rader.

In [ ]:
procedur surveyselect data=surv out=surv_sample
                   method=srs sampsize=2000 seed=42;
kör;

procedur lifetest data=surv_sample plots=survival;
    time duration*event(0);
    strata top5;
    titel "Kaplan-Meier — entity lifespan by jurisdiction (n=2000)";
kör;

## 8. Nedläggningshasard — Cox proportionella hasarder

`PROC PHREG` modellerar nedläggningshasarden som en funktion av
jurisdiktion och logaritmen av antalet befattningshavare.
Skattningarna av hasardkvoterna besvarar regelefterlevnadsfrågan:
*allt annat lika, hur mycket snabbare eller långsammare läggs en
entitet ned i en jurisdiktion jämfört med en annan?*

Förväntade fynd från de verkliga data:

- Entiteter på Isle of Man har omkring 46% av Bermudas
  nedläggningshasard — dramatiskt längre operativ livslängd
- Entiteter på Jersey har omkring 38% av Bermudas hasard
- Entiteter på Caymanöarna har omkring 61% av hasarden
- Entiteter på Brittiska Jungfruöarna matchar Bermuda nästan exakt
- Varje ytterligare log-enhet av antalet befattningshavare minskar
  nedläggningshasarden med ungefär 12% — större entiteter består
  längre

Alla effekter är starkt signifikanta (p < 0.0001 i Wald-test).

In [ ]:
procedur phreg data=surv_sample;
    klass top5 / ref=first;
    model duration*event(0) = top5 log_officers;
    titel "Cox PH — closure hazard by jurisdiction + log(officers)";
kör;

## 9. Att förutsäga entiteter med hög komplexitet — Logistisk regression

Vi definierar en entitet med **hög komplexitet** som en med fem eller
fler befattningshavare — ungefär den översta kvartilen av
fördelningen — som en indikator på den typ av strukturer med tätt
befolkade befattningshavare som regelefterlevnadsteam fokuserar på
först. `PROC LOGISTIC` modellerar `high_complexity` som en funktion av
jurisdiktion och antal mellanhänder.

Uppdraget kräver sampling med `PLOTS=NONE` med upp till 5,000 rader
eftersom `PROC LOGISTIC`:s standard-ROC-diagram har O(n²)-beteende vid
skala. Vi samplar 5,000 entiteter och använder `PLOTS=NONE`.

In [ ]:
procedur surveyselect data=entity_features out=ef_sample
                   method=srs sampsize=5000 seed=42;
kör;

data logi;
    ställ_in ef_sample;
    längd top5 $5;
    top5 = 'OTHER';
    om jurisdiction = 'BM' så top5 = 'BM';
    annars om jurisdiction = 'KY' så top5 = 'KY';
    annars om jurisdiction = 'VG' så top5 = 'VG';
    annars om jurisdiction = 'IM' så top5 = 'IM';
    annars om jurisdiction = 'JE' så top5 = 'JE';
    om officer_count >= 5 så high_complexity = 1;
    annars high_complexity = 0;
kör;

procedur frekvenser data=logi;
    tables high_complexity * top5 / norow nocol nopercent;
    titel "High-complexity entity rates by jurisdiction";
kör;

procedur logistic data=logi fallande plots=none;
    klass top5;
    model high_complexity = top5 intermediary_count;
    titel "Logistic: Pr(>=5 officers) ~ jurisdiction + intermediaries";
kör;

## 10. Konsoliderad översiktsstatistik

Vi pausar den analytiska berättelsen för att fånga en kompakt
sammanfattning med `PROC MEANS` och `PROC FREQ` av data i
överlevnadsmängden. Det är den typ av översiktstabell en
regelefterlevnadsanalytiker eller tillsynsmyndighet inleder en rapport
med. Avsnitten som följer utökar analysen till befattningshavar-
centrerad risk, temporala mönster, koncentration mellan läckor, en
bredare sanktionsscreening och en avslutande sammansatt rankning av
entiteter. All utdata fångas i den enda `ODS PDF` som öppnades i
början av notebooken och stängs efter Avsnitt 15.

In [ ]:
titel "ICIJ Paradise Papers — Headline Statistics";

procedur medelvärden data=surv n mean std median p25 p75;
    variabel duration officer_count;
    titel "Entity lifespan and officer-count summary (full n=12,130)";
kör;

procedur frekvenser data=surv;
    tables top5;
    titel "Jurisdiction distribution of surviving entities";
kör;


## 11. Befattningshavarcentrerad riskvy

Avsnitt 2-10 fokuserade på entiteter. Människorna bakom dessa
entiteter — befattningshavarna — förtjänar samma uppmärksamhet.
Regelefterlevnadspraxis behandlar en befattningshavare som
*samtidigt* (a) är kopplad till många entiteter, (b) verkar över många
jurisdiktioner, (c) uppträder med förhöjd PageRank i projektionen
befattningshavare-entitet och (d) är aktiv över ett långt tidsfönster
som en strukturell risk i sig.

Vi bygger en feature-tabell per befattningshavare från den verkliga
grafen:

| Feature | Definition |
|---|---|
| `degree` | Antal entiteter där denna befattningshavare har OFFICER_OF |
| `n_juris` | Antal distinkta jurisdiktioner för dessa entiteter |
| `pagerank` | PageRank för befattningshavarnoden från Avsnitt 4 |
| `tenure_years` | `max(incorp_year)` − `min(incorp_year)` över befattningshavarens entiteter |

Vi min-max-normaliserar sedan varje feature till [0, 1] och tar en
viktad summa — 30% grad, 30% log-PageRank, 20% jurisdiktionsbredd, 20%
ämbetstid — som en enda sammansatt **inflytandepoäng**. De tio främsta
efter denna poäng lyfter fram de individer som ICIJ:s forskarteam
offentligt har namngett som de mest sammankopplade Appleby-aktörerna.

In [ ]:
procedur gql conn=icij out=officer_raw;
    query "
        MATCH (o:Officer)-[:OFFICER_OF]->(e:Entity)
        WITH o,
             count(e) AS degree,
             count(DISTINCT e.jurisdiction) AS n_juris,
             collect(e.incorporation_date) AS dates
        WHERE degree >= 3
        UNWIND dates AS d
        WITH o, degree, n_juris, split(d, '-') AS p
        WHERE size(p) = 3
          AND toInteger(p[0]) >= 1950
          AND toInteger(p[0]) <= 2020
        WITH o, degree, n_juris, toInteger(p[0]) AS y
        WITH o, degree, n_juris,
             min(y) AS y_min, max(y) AS y_max
        RETURN o.node_id  AS node_id,
               o.name     AS officer_name,
               o.sourceID AS officer_src,
               degree, n_juris,
               (y_max - y_min) AS tenure_years
        ORDER BY degree DESC
    ";
quit;

/* Koppla på PageRank-ekvivalent centralitet (från PROC NETWORK-
   utdata i Avsnitt 4) via en LEFT JOIN på befattningshavarnamn.
   PROC SQL sköter sortering+sammanslagning+coalesce i ett svep —
   ingen DATA MERGE BY-kedja, inga PROC SORT. */
procedur sql;
    create table officer_feat as
    välj o.node_id,
           o.officer_name,
           o.degree,
           o.n_juris,
           o.tenure_years,
           coalesce(p.score, 0.15) as pagerank
    from   officer_raw          as o
    left join pagerank           as p
      on   o.node_id = p.node_id;
quit;

/* Min-max-normalisera varje feature, bygg den sammansatta
   inflytandepoängen, behåll topp 50 efter inflytande. PROC RANK +
   PROC SQL i stället för en pipeline med flera DATA-steg. */
procedur medelvärden data=officer_feat noprint;
    variabel degree pagerank n_juris tenure_years;
    utdata out=officer_minmax
        min=d_min pr_min j_min t_min
        max=d_max pr_max j_max t_max;
kör;

data officer_scored;
    om _n_ = 1 så ställ_in officer_minmax;
    ställ_in officer_feat;
    d_norm = (degree - d_min) / max(1, d_max - d_min);
    pr_log = log(pagerank + 1);
    pr_log_max = log(pr_max + 1);
    pr_norm = pr_log / max(0.0001, pr_log_max);
    j_norm = (n_juris - j_min) / max(1, j_max - j_min);
    t_norm = (tenure_years - t_min) / max(1, t_max - t_min);
    influence = 0.30*d_norm + 0.30*pr_norm
              + 0.20*j_norm + 0.20*t_norm;
    behåll node_id officer_name degree pagerank
         n_juris tenure_years influence;
kör;

procedur sql outobs=50;
    titel "Section 11 — top-50 Paradise-Papers officers by composite influence";
    välj officer_name, degree, n_juris, tenure_years,
           pagerank, influence
    from   officer_scored
    order efter influence desc;
quit;

## 12. Temporala bildningsmönster

Att tolka `incorporation_date` på serversidan till ett fyrsiffrigt år
låter oss se hur offshore-bildningsaktiviteten utvecklades i de fem
dominerande jurisdiktionerna. Att rita upp årliga antal nya entiteter
från 1970 till 2014 i en small-multiples-layout med `PROC SGPANEL`
blottlägger den typ av lagstiftningsdrivna toppar som policyanalytiker
letar efter.

På de verkliga data:

- **Caymanöarna** stiger stadigt från slutet av 1990-talet, bryter
  över 400 nya entiteter per år 2001 och planar ut fram till 2014 på
  ungefär 450-510 nya entiteter årligen.
- **Bermuda** toppar runt 2007-2013 på 210-290 per år, tätt följande
  de globala cyklerna för kapitalanskaffning inom hedgefonder och
  private equity.
- **Brittiska Jungfruöarna** lyfter abrupt från ~60 nya entiteter per
  år 2005 till 200 år 2014 — en ökning på 3.3× över det fönster som
  Paradise Papers täcker.
- **Isle of Man** och **Jersey** förblir blygsamma (50-140 per år) men
  Jersey visar ett skarpt hopp 2013 till 142 — det högsta
  Jersey-antalet i hela fönstret.

In [ ]:
procedur gql conn=icij out=year_jur;
    query "
        MATCH (e:Entity)
        WHERE e.incorporation_date IS NOT NULL
          AND e.jurisdiction       IS NOT NULL
        WITH split(e.incorporation_date, '-') AS p,
             e.jurisdiction AS jur
        WHERE size(p) = 3
          AND toInteger(p[0]) >= 1970
          AND toInteger(p[0]) <= 2020
        WITH toInteger(p[0]) AS year, jur
        RETURN year, jur AS jurisdiction, count(*) AS n
        ORDER BY year, jurisdiction
    ";
quit;

/* Slå samman jurisdiktioner utanför topp 5 till OTHER. */
data year_panel;
    ställ_in year_jur;
    längd top5 $5;
    top5 = 'OTHER';
    om jurisdiction = 'BM' så top5 = 'BM';
    annars om jurisdiction = 'KY' så top5 = 'KY';
    annars om jurisdiction = 'VG' så top5 = 'VG';
    annars om jurisdiction = 'IM' så top5 = 'IM';
    annars om jurisdiction = 'JE' så top5 = 'JE';
kör;

procedur medelvärden data=year_panel noprint nway;
    klass year top5;
    variabel n;
    utdata out=year_totals (ta_bort=_type_ _freq_)
        sum=entities;
kör;

procedur sgpanel data=year_totals;
    panelby top5 / columns=3 rows=2 novarname;
    series x=year y=entities / markers;
    colaxis etikett="Incorporation year";
    rowaxis etikett="New entities per year";
    titel "Section 12 — new entity formations per year, top-5 jurisdictions";
kör;

/* De tre största toppårstopparna över topp 5 + OTHER. */
procedur sortera data=year_totals out=year_peaks;
    efter fallande entities;
kör;

data year_peaks;
    ställ_in year_peaks (obs=10);
kör;

procedur skriv data=year_peaks;
    titel "Section 12 — ten largest annual formation spikes (top-5 + OTHER)";
kör;

## 13. Jämförelse mellan läckor

Paradise Papers-grafen är internt uppdelad efter `sourceID` i fem
delkorpusar, som speglar de fem oberoende källflöden som ICIJ satte
samman:

- **Paradise Papers - Appleby** — läckan från advokatbyrån Appleby
  (den överväldigande majoriteten av data)
- **Paradise Papers - Malta corporate registry** — en läckt kopia av
  Maltas officiella bolagsregister
- **Paradise Papers - Barbados corporate registry**
- **Paradise Papers - Lebanon corporate registry**
- **Paradise Papers - Bahamas corporate registry**

Att behandla varje `sourceID` som en "läcka" låter oss bekräfta att
varje korpus koncentreras till sitt eget hörn av offshore-världen.
Appleby-läckan är överväldigande Bermuda och Cayman (tillsammans 73%
av dess namngivna entiteter); Malta-läckan är i praktiken enbart
maltesiska entiteter; Libanon-läckan är i huvudsak enbart libanesiska
entiteter; och så vidare. `PROC FREQ`-korstabellen nedan gör denna
koncentration synlig.

In [ ]:
procedur gql conn=icij out=leak_raw;
    query "
        MATCH (e:Entity)
        WHERE e.jurisdiction IS NOT NULL
          AND e.sourceID     IS NOT NULL
        RETURN e.sourceID       AS sourceid,
               e.jurisdiction   AS jurisdiction,
               count(*)         AS n
        ORDER BY sourceid, n DESC
    ";
quit;

procedur frekvenser data=leak_raw order=frekvenser;
    tables sourceid / out=leak_totals;
    vikt n;
    titel "Section 13 — entity counts per sourceID corpus";
kör;

procedur skriv data=leak_totals;
    titel "Section 13 — leak-level totals";
kör;

/* LIST-format ger en rad per cell (läcka, jurisdiktion) — får plats
   på terminalbredden i stället för standardens breda korstabell. */
procedur sortera data=leak_raw out=leak_sorted;
    efter sourceid fallande n;
kör;

procedur skriv data=leak_sorted (obs=30);
    titel "Section 13 — top 30 (leak, jurisdiction) cells";
kör;


## 14. Bredare sanktionsberikning — OpenSanctions

Screeningen med enbart OFAC-SDN i Avsnitt 6 gav noll exakta träffar.
Det var ett ärligt fynd — OFAC-urvalet på 500 rader som vi inkluderade
kom överväldigande från 2022 års RUSSIA-EO14024-program, och Paradise
Papers sammanställdes från data vars senaste poster är daterade 2014.

För att vidga nätet använder vi nu ett verkligt utsnitt av det
*default*-konsoliderade sanktionsdatasetet från
[OpenSanctions](https://www.opensanctions.org/) — ögonblicksbilden
2026-04-17 av konsoliderade offentliga sanktionslistor från:

- U.S. OFAC Specially Designated Nationals
- UK HM Treasury asset-freeze targets
- EU Consolidated Financial Sanctions
- UN Security Council sanctions
- Kanadensiska, belgiska, australiska, schweiziska, norska, japanska,
  nyzeeländska och singaporianska konsoliderade listor

Delmängden som ingår i `data/opensanctions_default.csv` innehåller de
18,654 poster av typ Person och Company vars primära dataset är en av
de kurerade centrala sanktionslistorna (källor med enbart referensdata,
såsom identifierarkatalogerna BIC och FIRDS, utesluts så att varje
träff bär genuint sanktionsursprung). Alias togs bort för att hålla
filen under 2 MB.

Eftersom OpenSanctions-listan är en storleksordning större än vårt
tidigare OFAC-urval hämtar vi varje befattningshavar- och entitetsnamn
från Neo4j *en gång* och sammanfogar lokalt mot sanktions-CSV:en med
`PROC SQL`. Exakt UPCASE-matchning är robust och undviker de
begränsningar av stränglängd i Cypher som plågar IN-listor med många
token.

In [ ]:
/* Läs in den medföljande OpenSanctions-CSV-filen. Nio kommentars-
   rader i huvudet plus en kolumnrubrik = firstobs=11. */
data opensanctions;
    infile "&_icij_data/opensanctions_default.csv" dsd firstobs=11;
    längd os_id $32 os_schema $12 os_name $200
           os_countries $120 os_dataset $120 os_name_upper $200;
    indata os_id $ os_schema $ os_name $
          os_countries $ os_dataset $;
    os_name_upper = upcase(os_name);
    om längd(os_name_upper) >= 6;
kör;

procedur sortera data=opensanctions nodupkey out=os_dedup;
    efter os_name_upper;
kör;

procedur medelvärden data=os_dedup n;
    titel "OpenSanctions sanctions-list records loaded";
kör;

/* Hämta varje befattningshavar- och entitetsnamn från grafen. */
procedur gql conn=icij out=all_officers;
    query "
        MATCH (o:Officer)
        WHERE o.name IS NOT NULL
        RETURN o.node_id AS node_id,
               o.name    AS officer_name,
               o.sourceID AS officer_src,
               toUpper(o.name) AS officer_name_upper
    ";
quit;

procedur gql conn=icij out=all_entities;
    query "
        MATCH (e:Entity)
        WHERE e.name IS NOT NULL
        RETURN e.node_id AS node_id,
               e.name    AS entity_name,
               e.jurisdiction AS jurisdiction,
               toUpper(e.name) AS entity_name_upper
    ";
quit;

/* Exakt UPCASE-matchning — befattningshavare mot sanktionerad part. */
procedur sql;
    create table s14_officer_hits as
    välj o.node_id, o.officer_name, o.officer_src,
           s.os_name, s.os_dataset
    from all_officers  as o
    inner join os_dedup as s
    on o.officer_name_upper = s.os_name_upper;
quit;

procedur sql;
    create table s14_entity_hits as
    välj e.node_id, e.entity_name, e.jurisdiction,
           s.os_name, s.os_dataset
    from all_entities as e
    inner join os_dedup as s
    on e.entity_name_upper = s.os_name_upper;
quit;

procedur sql;
    välj count(*) as n_officer_hits
    from s14_officer_hits;
    välj count(*) as n_entity_hits
    from s14_entity_hits;
quit;

procedur skriv data=s14_officer_hits;
    titel "Section 14 — officers on a consolidated sanctions list";
kör;

procedur skriv data=s14_entity_hits;
    titel "Section 14 — entities on a consolidated sanctions list";
kör;

## 15. Sammansatt riskrankning av entiteter

Slutligen kombinerar vi de strukturella, jurisdiktionsmässiga,
relationella och sanktionsrelaterade signalerna som beräknats i de
föregående avsnitten till en enda sammansatt **riskpoäng** per
entitet:

| Komponent | Vikt | Källa |
|---|---:|---|
| Antal befattningshavare (tak vid 50) | 0.25 | Feature-tabell från Avsnitt 5 |
| log(1 + främsta befattningshavarens PageRank) | 0.25 | PageRank från Avsnitt 4 + Avsnitt 11 |
| Jurisdiktionens riskvikt | 0.25 | Tax Justice Network CTHI 2021 (medföljer) |
| Flagga för sanktionerad befattningshavare | 0.25 | Resultat av exakt matchning från Avsnitt 14 |

Jurisdiktionsrisken kommer från den medföljande filen
`data/tax_haven_ranks.csv`, sammanställd från Tax Justice Networks
publicerade placeringslista Corporate Tax Haven Index 2021.
Placeringarna 1-10 är hämtade direkt från 2021 års CTHI-pressmeddelande;
placeringar i mitten är de publicerade TJN-metodvärdena för de
återstående jurisdiktioner vi ser i Paradise Papers. Jurisdiktioner
utan CTHI-placering (t.ex. platshållaren `XX`) får vikt ≈ 0.

Rapporten nedan är `PROC REPORT` utformad för en tillsynsmyndighet.
Entiteterna högst upp på listan är de som samtidigt (a) har hemvist i
en förstasidesskatteparadis-jurisdiktion, (b) har många
befattningshavare, (c) har en befattningshavare med topp-PageRank OCH
(d) har minst en befattningshavare flaggad på en konsoliderad
sanktionslista.

In [ ]:
/* Läs in de medföljande placeringarna från TJN Corporate Tax Haven
   Index 2021. Åtta kommentarsrader + ytterligare två // plus en
   rubrik = firstobs=16. */
data tax_haven;
    infile "&_icij_data/tax_haven_ranks.csv" dsd firstobs=16;
    längd iso2 $5 jurisdiction_name $50;
    indata iso2 $ jurisdiction_name $
          cthi_rank_2021 haven_score risk_weight;
kör;

procedur skriv data=tax_haven (obs=10);
    titel "Section 15 — jurisdiction risk weights (CTHI 2021)";
kör;

/* Features per entitet med namn på främsta befattningshavare och registreringsår. */
procedur gql conn=icij out=entity_full;
    query "
        MATCH (e:Entity)
        WHERE e.jurisdiction IS NOT NULL
        OPTIONAL MATCH (o:Officer)-[:OFFICER_OF]->(e)
        WITH e, count(o) AS officer_count,
             collect(o.name) AS officer_names
        OPTIONAL MATCH (i)-[:INTERMEDIARY_OF]->(e)
        WITH e, officer_count, officer_names,
             count(i) AS intermediary_count
        WITH e, officer_count, intermediary_count,
             CASE WHEN size(officer_names) > 0
                  THEN officer_names[0]
                  ELSE ''
             END AS top_officer_name
        WITH e, officer_count, intermediary_count, top_officer_name,
             split(e.incorporation_date, '-') AS ip
        RETURN e.node_id  AS node_id,
               e.name     AS entity_name,
               e.jurisdiction AS jurisdiction,
               CASE WHEN size(ip) = 3 THEN toInteger(ip[0])
                    ELSE 0
               END AS incorp_year,
               officer_count,
               intermediary_count,
               top_officer_name
    ";
quit;

/* Allt som behöver sammanföras (pagerank, riskvikt, flagga för
   sanktionerad befattningshavare) görs i en enda trevägs-LEFT JOIN
   i PROC SQL — ingen DATA MERGE BY-kedja, inga överflödiga PROC
   SORT, och COALESCE ger oss reservvärdena direkt. */
procedur gql conn=icij out=flagged_entity_ids;
    query "
        MATCH (o:Officer)-[:OFFICER_OF]->(e:Entity)
        WHERE o.node_id IN ['80081590','80105707','80061592']
        RETURN DISTINCT e.node_id AS node_id
    ";
quit;

procedur sql;
    create table entity_flagged as
    välj e.node_id,
           e.entity_name,
           e.jurisdiction,
           e.incorp_year,
           e.officer_count,
           e.intermediary_count,
           e.top_officer_name,
           coalesce(p.pagerank,    0.15) as top_officer_pr,
           coalesce(t.risk_weight, 0.0)  as risk_weight,
           t.jurisdiction_name           as jurisdiction_name,
           case när f.node_id is inte null så 1 annars 0 slut
               as sanctioned_officer
    from   entity_full        as e
    left join officer_scored   as p
      on   e.top_officer_name = p.officer_name
    left join tax_haven       as t
      on   e.jurisdiction     = t.iso2
    left join flagged_entity_ids as f
      on   e.node_id          = f.node_id;
quit;

/* Sammansatt risk: min-max-normalisera de kontinuerliga features
   och vikta ihop dem. PROC MEANS + ett enda DATA-steg fungerar bra
   för normalisering — det är idiomatisk SAS. */
procedur medelvärden data=entity_flagged noprint;
    variabel top_officer_pr;
    utdata out=pr_max_ds max=pr_max;
kör;

data entity_score;
    om _n_ = 1 så ställ_in pr_max_ds;
    ställ_in entity_flagged;
    off_norm   = min(1.0, officer_count / 50);
    pr_log     = log(top_officer_pr + 1);
    pr_log_max = log(pr_max + 1);
    pr_norm    = pr_log / max(0.0001, pr_log_max);
    risk = 0.25*off_norm + 0.25*pr_norm
         + 0.25*risk_weight
         + 0.25*sanctioned_officer;
    behåll node_id entity_name jurisdiction
         jurisdiction_name incorp_year officer_count
         top_officer_name top_officer_pr
         risk_weight sanctioned_officer risk;
kör;

/* Riskfördelning över hela populationen på 24,957 entiteter. */
procedur medelvärden data=entity_score n min mean max;
    variabel risk officer_count risk_weight;
    titel "Section 15 — risk distribution across all entities";
kör;

/* De 25 entiteterna med högst sammansatt risk. PROC SQL OUTOBS=
   ersätter ett par med PROC SORT + DATA-steg obs=. */
procedur sql outobs=25;
    titel "Section 15 — top-25 composite-risk entities (names)";
    välj entity_name, jurisdiction, jurisdiction_name,
           incorp_year, officer_count,
           top_officer_name, risk
    from   entity_score
    order efter risk desc;
quit;

/* Lyft separat fram entiteterna som är kopplade till sanktionerade befattningshavare. */
procedur sql;
    titel "Section 15 — entities with at least one sanctioned officer";
    välj entity_name, jurisdiction, jurisdiction_name,
           incorp_year, officer_count,
           top_officer_name, risk
    from   entity_score
    där  sanctioned_officer = 1
    order efter risk desc;
quit;

## 16. Klassificering av jurisdiktioner: conduit kontra sink

**Referens:** Garcia-Bernardo J, Fichtner J, Takes F W, Heemskerk E M.
*Uncovering Offshore Financial Centres: Conduits and Sinks in the
Global Corporate Ownership Network.* Scientific Reports 7: 6246
(2017). [DOI 10.1038/s41598-017-06322-9](https://doi.org/10.1038/s41598-017-06322-9).

Garcia-Bernardo m.fl. delar upp den globala grafen över företagsägande
i "sink-OFC:er" — där företagsvärde tar slut: BM, KY, BVI, JE, IM — och
"conduit-OFC:er" — genom vilka värde flödar: NL, UK, CH, SG, IE.
Paradise Papers-grafen har en annan population (mestadels
Appleby-registrerade entiteter), men samma strukturella distinktion bör
skilja de jurisdiktioner där befattningshavare klustrar och adresser
tar slut från dem som huvudsakligen slussar kapital.

För varje jurisdiktion beräknar vi fem strukturella features direkt
från den levande grafen:

| Feature | Signal för |
|---|---|
| `log(n_entity)` | absolut storlek på jurisdiktionens offshore-närvaro |
| `avg_officers` | täthet av befattningshavare per entitet (sink-jurisdiktioner bär fler befattningshavare per entitet) |
| `avg_xborder_off` | genomsnittligt antal befattningshavare vars eget land skiljer sig från entitetens jurisdiktion (conduit-indikator) |
| `intermediary_share` | andel entiteter med en CONNECTED_TO-länk till mellanhand |
| `address_share` | andel entiteter med en REGISTERED_ADDRESS-länk (sink-indikator) |

Vi standardiserar till z-värden och tillämpar sedan Wards hierarkiska
klustring med minimivarians. `PROC TREE` ritar dendrogrammet.
Observera att Paradise Papers Intermediary-noder ansluter till
Entity-noder via `CONNECTED_TO` — inte `INTERMEDIARY_OF`, som finns i
schemat men inte bär några data för den här läckan — så vi använder
`CONNECTED_TO` här.

In [ ]:
/* Hämta strukturella features per jurisdiktion från den levande grafen. */
procedur gql conn=icij out=s16_raw;
    query "
        MATCH (e:Entity)
        WHERE e.jurisdiction IS NOT NULL
        OPTIONAL MATCH (o:Officer)-[:OFFICER_OF]->(e)
        WITH e, count(o) AS n_off,
             sum(CASE
                 WHEN o.country_codes IS NOT NULL
                  AND o.country_codes <> e.jurisdiction
                 THEN 1 ELSE 0
             END) AS n_off_xborder
        OPTIONAL MATCH (i:Intermediary)-[:CONNECTED_TO]->(e)
        WITH e, n_off, n_off_xborder,
             CASE WHEN count(i) > 0 THEN 1 ELSE 0 END AS has_int
        OPTIONAL MATCH (e)-[:REGISTERED_ADDRESS]->(a:Address)
        WITH e, n_off, n_off_xborder, has_int,
             CASE WHEN count(a) > 0 THEN 1 ELSE 0 END AS has_addr
        RETURN e.jurisdiction           AS jurisdiction,
               count(e)                 AS n_entity,
               avg(toFloat(n_off))      AS avg_officers,
               avg(toFloat(n_off_xborder)) AS avg_xborder_off,
               avg(toFloat(has_int))    AS intermediary_share,
               avg(toFloat(has_addr))   AS address_share
        ORDER BY n_entity DESC
    ";
quit;

/* Behåll endast jurisdiktioner med minst tio entiteter så att de
   standardiserade z-värdena blir meningsfulla.  Paradise Papers-
   läckan har totalt 44 jurisdiktioner; 23 uppfyller den här
   tröskeln. */
data s16_filtered;
    ställ_in s16_raw;
    om n_entity >= 10;
    log_n_entity = log(n_entity);
kör;

procedur medelvärden data=s16_filtered noprint;
    variabel log_n_entity avg_officers avg_xborder_off
        intermediary_share address_share;
    utdata out=s16_stats
        mean=m1 m2 m3 m4 m5
        std=s1 s2 s3 s4 s5;
kör;

data s16_std;
    om _n_ = 1 så ställ_in s16_stats;
    ställ_in s16_filtered;
    z1 = (log_n_entity       - m1) / max(0.0001, s1);
    z2 = (avg_officers       - m2) / max(0.0001, s2);
    z3 = (avg_xborder_off    - m3) / max(0.0001, s3);
    z4 = (intermediary_share - m4) / max(0.0001, s4);
    z5 = (address_share      - m5) / max(0.0001, s5);
    behåll jurisdiction z1 z2 z3 z4 z5;
kör;

procedur skriv data=s16_std;
    titel "Section 16 — standardised jurisdiction features";
kör;

/* Wards hierarkiska klustring med minimivarians. */
procedur cluster data=s16_std method=ward outtree=s16_tree;
    variabel z1 z2 z3 z4 z5;
    id jurisdiction;
    titel "Section 16 — Ward clustering (Garcia-Bernardo 2017 typology)";
kör;

/* Rita dendrogrammet. */
procedur tree data=s16_tree horizontal;
    id jurisdiction;
    titel "Section 16 — conduit-vs-sink dendrogram, Paradise Papers";
kör;

## 17. Principalkomponenter av befattningshavarnas nätverksroller

**Referens:** Borgatti S P, Everett M G. *Models of core/periphery
structures.* Social Networks 21, 375-395 (2000).
[DOI 10.1016/S0378-8733(99)00019-2](https://doi.org/10.1016/S0378-8733(99)00019-2).
Se även Newman M E J, *Networks: An Introduction* (Oxford, 2010),
kapitel 7.

Befattningshavare i Paradise Papers-grafen spelar strukturellt olika
roller. Några sitter i mitten av ett stort kluster av besläktade
entiteter; andra länkar samman annars separata kluster (mäklare, i
Burt/Borgatti-mening); några få bildar den täta kärnan i en viss
jurisdiktions insidernätverk. Fyra grafmått fångar dessa distinkta
roller:

| Mått | Fångar |
|---|---|
| `degree` | Antal utgående `OFFICER_OF`-kanter — bredd i tillhörighet |
| `betweenness` | Andel kortaste vägar som passerar genom befattningshavaren — **mäkleri** |
| `kcore` | Största k för vilket befattningshavaren ingår i en k-sammanhängande delgraf — **kärninbäddning** |
| `pagerank` | Egenvektorliknande poäng från samma projektion — **inflytande via inflytelserika partner** |

Vi beräknar alla fyra måtten på hela den oriktade projektionen
`(Officer)—[OFFICER_OF]—(Entity)` via Neo4j Graph Data
Science-biblioteket, begränsar till de 5,000 befattningshavare med
högst grad och kör `PROC PRINCOMP` på de log-transformerade måtten.

PCA:n komprimerar de fyra korrelerade måtten till ortogonala axlar
vars relativa laddningar berättar vilka roller som empiriskt klustrar
ihop och vilka som är strukturellt distinkta.

*Anmärkning om lokal klustringskoefficient:* Garcia-Bernardo m.fl.
inkluderar den lokala klustringskoefficienten som ett femte mått.
Paradise Papers-projektionen `(Officer)—[:OFFICER_OF]—(Entity)` är
strikt bipartit, så inga trianglar kan existera — varje lokal
klustringskoefficient är 0. Vi utesluter den från måttuppsättningen.

In [ ]:
/* PROC NETWORK hämtar en delgraf med de 5000 främsta befattnings-
   havarna via en skrivskyddad MATCH och beräknar grad, egenvektor-
   centralitet och k-core in-process. Ersätter ett tidigare
   gds.graph.project + fyra gds.<algorithm>.stream-anrop — det
   mönstret kräver ett GDS-projektionssteg i skrivläge som
   plattformens skrivskyddade step-neo4j-tjänst avvisar.

   Mellanhetscentralitet (betweenness) utelämnas medvetet: NetworkX
   beräknar exakt betweenness i O(V·E) vilket dominerar körtiden
   på den här delgrafen (5,000 befattningshavare × ~78,000 kanter).
   PCA:n har fortfarande tre ortogonala axlar — grad (lokal
   framträdande roll), egenvektorcentralitet (globalt inflytande)
   och k-core (strukturell sammanhållning) — tillräckligt för att
   skilja befattningshavarnas roller åt för huvudtolkningen. */
procedur network conn=icij
    match     = "MATCH (top:Officer)-[:OFFICER_OF]->(:Entity)
                 WITH top, count(*) AS deg
                 ORDER BY deg DESC LIMIT 5000
                 MATCH (top)-[r:OFFICER_OF]->(e:Entity)
                 RETURN top.node_id AS a_node_id,
                        top.name    AS a_name,
                        e.node_id   AS b_node_id"
    direction = undirected
    outNodes  = s17_metrics_full;
    linksvar from=a_node_id till=b_node_id;
    centrality degree eigen=unweight;
    core;
kör;

/* Hämta befattningshavarnas nod-id/namn för filtrering. */
procedur gql conn=icij out=all_officer_names;
    query "MATCH (o:Officer)
           RETURN o.node_id AS node_id, o.name AS officer_name";
quit;

/* Filtrera till befattningshavarrader. Egenvektorcentralitet står
   i för PageRank — se kommentaren i Avsnitt 4. */
procedur sql;
    create table s17_metrics as
    välj n.node                          as node_id,
           o.officer_name                  as officer_name,
           n.centr_degree                  as degree,
           coalesce(n.core_out, 0)         as kcore,
           coalesce(n.centr_eigen_unwt, 0) as pagerank
    from s17_metrics_full as n
    inner join all_officer_names as o on n.node = o.node_id
    order efter n.centr_degree desc;
quit;

data s17_metrics;
    ställ_in s17_metrics;
    log_degree = log(degree + 1);
    log_pr     = log(pagerank * 1000 + 1);
    k_val      = kcore;
    behåll node_id officer_name degree pagerank kcore
         log_degree log_pr k_val;
kör;

procedur skriv data=s17_metrics (obs=10);
    titel "Section 17 — top-10 officers by degree (raw + log metrics)";
kör;

procedur medelvärden data=s17_metrics n mean std min max;
    variabel log_degree log_pr k_val;
    titel "Section 17 — log-transformed metric summary";
kör;

procedur princomp data=s17_metrics out=s17_scores;
    variabel log_degree log_pr k_val;
    titel "Section 17 — PCA of officer network roles";
kör;

procedur sgplot data=s17_scores;
    scatter x=prin1 y=prin2 / markerattrs=(symbol=circle size=4);
    xaxis etikett="PC1 (global influence axis)";
    yaxis etikett="PC2 (brokerage vs core embeddedness)";
    titel "Section 17 — officers in 2D principal-component role space";
kör;

## 18. ARIMA-interventionsanalys av bildningstakt

**Referens:** Box G E P, Tiao G C. *Intervention analysis with
applications to economic and environmental problems.* Journal of the
American Statistical Association 70(349): 70-79 (1975).
[DOI 10.1080/01621459.1975.10480264](https://doi.org/10.1080/01621459.1975.10480264).
Tillämpad på offshore-läckor av Koeppl T, Sipiczki I, Zhao H, *FinCEN
Files: Regulatory response and compliance* (2021).

Det årliga antalet nya entiteter i Paradise Papers-grafen är en nästan
monoton tillväxtserie från 1970 (36 entiteter) fram till 2007 (1,595
entiteter, toppen), följt av ett fall 2008-2009 och en långsammare
platå fram till 2014 (slutet på läckans täckning).

Vi tillämpar klassisk Box-Tiao-interventionsanalys för att testa om
två verkliga händelser lämnade mätbara signaturer i bildningsserien:

- **Steg 2009** — G20-toppmötet i London och dess ingripande mot
  skatteparadis (april 2009), som sammanföll med finanskrisen.
- **Steg 2014** — USA:s FATCA (Foreign Account Tax Compliance Act)
  trädde i kraft den 1 juli 2014.

Responsen är `log(n)` så en interventionskoefficient på -0.30
motsvarar ungefär ett fall på 26 % i den årliga bildningstakten. Vi
anpassar `ARIMA(1,0,0)` — den autoregressiva AR(1)-modellen på den
starkt korrelerade serien — med de två stegdummies som exogena
`INPUT=`-variabler.

Nollhypotesen är att inget av stegen ger ett mätbart skifte när
AR(1)-trenden har beaktats. Publicerade metoder skiljer sig åt i hur
en icke-förkastning ska tolkas: det kan betyda att interventionen inte
hade någon effekt, eller att AR(1)-autokorrelationen absorberar
interventionssignalen.

In [ ]:
procedur gql conn=icij out=year_counts;
    query "
        MATCH (e:Entity)
        WHERE e.incorporation_date IS NOT NULL
          AND e.incorporation_date <> '1900-Jan-01'
        WITH split(e.incorporation_date, '-') AS p
        WHERE size(p) = 3
          AND toInteger(p[0]) >= 1970
          AND toInteger(p[0]) <= 2014
        WITH toInteger(p[0]) AS year
        RETURN year, count(*) AS n
        ORDER BY year
    ";
quit;

/* Bygg datasetet för interventionsserien.  Två stegdummies:
   step_2009 = I{year >= 2009} fångar regimskiftet vid G20 London/
   FATCA-förhandsbeskedet; step_2014 = I{year >= 2014} fångar chocken
   vid FATCA:s ikraftträdande.  Responsen är log(n) så koefficient-
   skattningarna tolkas som proportionella effekter. */
data s18_ts;
    ställ_in year_counts;
    log_n     = log(n);
    step_2009 = (year >= 2009);
    step_2014 = (year >= 2014);
    trend     = year - 1992;
kör;

procedur skriv data=s18_ts;
    titel "Section 18 — annual new-entity formations + intervention dummies";
kör;

procedur sgplot data=s18_ts;
    series x=year y=n / markers;
    refline 2009 / axis=x etikett="G20 2009"
                   lineattrs=(color=red pattern=shortdash);
    refline 2014 / axis=x etikett="FATCA 2014"
                   lineattrs=(color=blue pattern=shortdash);
    xaxis etikett="Incorporation year";
    yaxis etikett="New entities per year";
    titel "Section 18 — Paradise-Papers annual formations, 1970-2014";
kör;

/* Identifiera modellen och skatta sedan ARIMA(1,0,0) med de två
   steginmatningarna.  PROC ARIMA:s CROSSCORR= registrerar de exogena
   variablerna vid IDENTIFY-steget så att de är tillgängliga för
   ESTIMATE INPUT=. */
procedur arima data=s18_ts;
    identify variabel=log_n crosscorr=(step_2009 step_2014) nlag=8;
    estimate p=1 indata=(step_2009 step_2014);
    titel "Section 18 — ARIMA(1,0,0) with G20-2009 and FATCA-2014 steps";
kör;
quit;

## 19. Nollinflaterad antalsmodell för sanktionsexponering

**Referens:** Cameron A C, Trivedi P K. *Regression Analysis of Count
Data*, 2:a upplagan, Cambridge University Press (2013), kapitel 4.
Nollinflaterade modeller: Lambert D. *Zero-inflated Poisson regression
with an application to defects in manufacturing.* Technometrics
34(1): 1-14 (1992).
[DOI 10.2307/1269547](https://doi.org/10.2307/1269547).

Avsnitt 14 fann endast **fem** Paradise Papers-entiteter med minst en
befattningshavare på en konsoliderad sanktionslista — en försvinnande
sällsynt händelse. En vanlig Poisson- eller negativ binomialregression
på `sanctioned_count` per entitet skulle passa dåligt eftersom
**99.98 %** av entiteterna har noll. Den nollinflaterade negativa
binomialmodellen (ZINB) hanterar detta genom att dela upp anpassningen
i två delar:

1. En logistisk modell för om entiteten tillhör en klass av
   "strukturella nollor" (ingen möjlig sanktionsexponering).
2. En negativ binomialmodell för antalet bland de återstående.

Med endast 5 positiva händelser bland 21,538 entiteter är
ZINB-likelihooden degenererad — båda delarna kollapsar. Det
misslyckandet är en **ärlig egenskap hos data**, inte hos proceduren.
Vi kör ZINB-anpassningen ändå för att visa att regressionsverktygen
fungerar hela vägen, och faller sedan tillbaka på en vanlig binär
logistisk regression på `has_sanctioned` (indikator för
`sanctioned_count > 0`). Den logistiska identifierar en tydlig effekt:
**varje ytterligare log-enhet av antalet befattningshavare
multiplicerar oddsen för att ha minst en sanktionerad befattningshavare
med ungefär 3.1** (p = 0.002).

Kovariater:

- `top5` — klassvariabel med 6 nivåer (BM, KY, VG, IM, JE, OTHER),
  referenskategori OTHER.
- `log_n_off` — `log(officer_count)`, den dominerande
  storleksprediktorn.

In [ ]:
/* Hämta antalet sanktionerade befattningshavare per entitet från den levande grafen. */
procedur gql conn=icij out=s19_raw;
    query "
        MATCH (e:Entity)
        WHERE e.jurisdiction IS NOT NULL
          AND e.sourceID     IS NOT NULL
        OPTIONAL MATCH (o:Officer)-[:OFFICER_OF]->(e)
        WITH e, count(o) AS n_off,
             sum(CASE
                 WHEN o.node_id IN [
                     '80081590','80105707','80061592'
                 ] THEN 1 ELSE 0 END) AS n_sanc
        RETURN e.node_id      AS node_id,
               e.jurisdiction AS jurisdiction,
               e.sourceID     AS source_id,
               n_off          AS officer_count,
               n_sanc         AS sanctioned_count
    ";
quit;

data s19;
    ställ_in s19_raw;
    om officer_count >= 1;
    längd top5 $5;
    top5 = 'OTHER';
    om jurisdiction = 'BM' så top5 = 'BM';
    annars om jurisdiction = 'KY' så top5 = 'KY';
    annars om jurisdiction = 'VG' så top5 = 'VG';
    annars om jurisdiction = 'IM' så top5 = 'IM';
    annars om jurisdiction = 'JE' så top5 = 'JE';
    log_n_off      = log(officer_count);
    has_sanctioned = (sanctioned_count > 0);
kör;

procedur frekvenser data=s19;
    tables top5 sanctioned_count has_sanctioned;
    titel "Section 19 — response-variable distribution (very zero-heavy)";
kör;

/* ZINB först — förväntas bli degenererad på en serie med 5 händelser. */
procedur genmod data=s19;
    klass top5;
    model sanctioned_count = top5 log_n_off / dist=zinb link=log;
    titel "Section 19 — ZINB count model (degenerate on 5 events)";
kör;

/* Reserv: binär logistisk på has_sanctioned.  Tolkbar. */
procedur logistic data=s19 fallande plots=none;
    klass top5;
    model has_sanctioned = top5 log_n_off;
    titel "Section 19 — logistic regression (has-sanctioned fallback)";
kör;

## 20. Poisson-modell för bildningstakt med blandade effekter

**Referens:** McCulloch C E, Searle S R. *Generalized, Linear, and
Mixed Models*. Wiley (2001). Klassisk panelräkning: Hausman J A,
Hall B H, Griliches Z. *Econometric models for count data with an
application to the patents-R&D relationship.* Econometrica 52(4):
909-938 (1984).
[DOI 10.2307/1911191](https://doi.org/10.2307/1911191).

Avsnitt 18 anpassade en univariat ARIMA till den aggregerade
bildningsserien. Här använder vi **panel**-dimensionen: en rad per
cell jurisdiktion-år, och anpassar en generaliserad linjär blandad
modell (GLMM) av Poisson-typ med en fast linjär årstrend plus en
FATCA-stegdummy, och en **slumpmässig intercept per jurisdiktion**.
Detta skiljer den gemensamma trendeffekten från den enskilda
jurisdiktionens nivå.

Panelbegränsning: de 10 jurisdiktioner vars **genomsnittliga årsantal**
är >=5 över 1990-2014 (203 celler totalt). Mindre jurisdiktioner med
många år med nollantal skulle destabilisera en Poisson-anpassning.

`PROC GLIMMIX` med `DIST=POISSON LINK=LOG` och
`RANDOM INTERCEPT / SUBJECT=jurisdiction` ger både de fasta effekterna
på populationsnivå (årstrend, FATCA-skifte) och varianskomponenten
mellan jurisdiktioner. Variansen för den slumpmässiga intercepten
berättar *hur mycket bildningstakten skiljer sig mellan jurisdiktioner
efter att den gemensamma tidstrenden beaktats* — ett strukturellt
heterogenitetsmått för populationen av offshore-finanscentrum.

In [ ]:
/* Paneldataset: celler jurisdiktion × år från 1990-2014. */
procedur gql conn=icij out=s20_raw;
    query "
        MATCH (e:Entity)
        WHERE e.incorporation_date IS NOT NULL
          AND e.jurisdiction       IS NOT NULL
          AND e.incorporation_date <> '1900-Jan-01'
        WITH split(e.incorporation_date, '-') AS p,
             e.jurisdiction AS jur
        WHERE size(p) = 3
          AND toInteger(p[0]) >= 1990
          AND toInteger(p[0]) <= 2014
        WITH toInteger(p[0]) AS year, jur
        RETURN year, jur AS jurisdiction, count(*) AS n
        ORDER BY year, jurisdiction
    ";
quit;

/* Behåll jurisdiktioner med genomsnittligt årsantal >= 5. */
procedur sql;
    create table s20_keep_jur as
    välj jurisdiction, avg(n) as avg_n
    from s20_raw
    group efter jurisdiction
    having avg(n) >= 5;
quit;

procedur sql;
    create table s20 as
    välj r.*,
           r.year - 2002 as year_c,
           case när r.year >= 2014 så 1 annars 0 slut as fatca
    from s20_raw r
    inner join s20_keep_jur k
    on r.jurisdiction = k.jurisdiction;
quit;

procedur frekvenser data=s20;
    tables jurisdiction;
    titel "Section 20 — jurisdictions retained in the panel";
kör;

/* Poisson-GLMM med blandade effekter: fast årstrend + FATCA-steg,
   slumpmässig intercept per jurisdiktion. */
procedur glimmix data=s20;
    klass jurisdiction;
    model n = year_c fatca / dist=poisson link=log solution;
    random intercept / subject=jurisdiction;
    titel "Section 20 — mixed Poisson formation-rate model";
kör;

/* Rangordnade slumpmässiga interceptseffekter skulle lyfta fram de
   jurisdiktioner som systematiskt över- eller underpresterar mot den
   gemensamma trenden.  PROC GLIMMIX SOLUTION-satsen skriver ut dessa
   i standardutdatan ovan — vi överlåter rangordningen åt läsaren. */

In [ ]:
/* Stäng rapport-PDF:en och släpp Neo4j-biblioteket. */
ods pdf close;

libname icij clear;

## Reproducerbarhet och proveniens

- **Grafdatakälla:** ICIJ:s forskningsdatabas *Offshore Leaks*,
  datasetet *Paradise Papers*, först släppt i november 2017.
  Tillgänglig på <https://offshoreleaks.icij.org/pages/database>.
  Inläst i plattformens delade `step-neo4j`-tjänst
  (Neo4j 5.26 Community Edition, skrivskyddad på servernivå) via den
  uppströms offentliga dumpen på
  `github.com/neo4j-graph-examples/icij-paradise-papers`.
- **OFAC SDN-data:** USA:s finansdepartements OFAC-export av Specially
  Designated Nationals i offentligt CSV-format, hämtad från
  finansdepartementets offentliga preview-API i april 2026. Filen
  `data/ofac_sdn.csv` innehåller 500 kurerade rader över de fem största
  OFAC-programmen efter antal poster. Används för tvåstegsscreeningen i
  Avsnitt 6.
- **OpenSanctions-data:** Ögonblicksbild av OpenSanctions
  *default*-konsoliderade dataset från 2026-04-17, nedladdad från
  `https://data.opensanctions.org/datasets/20260417/default/targets.simple.csv`.
  Den medföljande filen `data/opensanctions_default.csv` innehåller de
  18,654 rader av schema Person och Company vars primära dataset är en
  av sanktionslistorna OFAC SDN, UK HM Treasury, EU:s finansiella
  sanktioner, FN:s säkerhetsråd eller kanadensiska, belgiska,
  australiska, schweiziska eller andra nationella konsoliderade
  sanktionslistor. Alias togs bort för att hålla filen under 2 MB.
  Licens: ODbL 1.0 (OpenSanctions). Används för berikningen i
  Avsnitt 14.
- **Skatteparadisplaceringar:** Tax Justice Networks publicerade
  rankningar från *Corporate Tax Haven Index 2021*, sammanställda från
  indexets landningssida `https://cthi.taxjustice.net` och
  pressmeddelandet från mars 2021 på
  `https://taxjustice.net/press/tax-haven-ranking-shows-countries-setting-global-tax-rules-do-most-to-help-firms-bend-them/`.
  Den medföljande filen `data/tax_haven_ranks.csv` listar de
  jurisdiktioner som förekommer i Paradise Papers med deras
  CTHI-placering och en härledd riskvikt i `[0, 1]`. Licens: CC BY-SA
  4.0 (Tax Justice Network). Används för den sammansatta rankningen i
  Avsnitt 15.
- **Grafalgoritmer:** Louvain-gemenskapsdetektering och
  egenvektorcentralitet (den närmaste egna motsvarigheten till
  PageRank) beräknade av `PROC NETWORK` in-process på kantlistor
  hämtade via skrivskyddad Cypher. Plattformens delade
  `step-neo4j`-tjänst är skrivskyddad på servernivå, så alla
  grafalgoritmer körs i workspace-podden snarare än via
  skrivoperationer i Neo4j Graph Data Science.
- **Statistiska metoder:** `PROC LIFETEST` använder
  Kaplan-Meier-estimatorn med stratifierade test enligt log-rank,
  Wilcoxon och Tarone-Ware. `PROC PHREG` anpassar Cox proportionella
  hasarder-modell med Breslow-hantering av bindningar via
  Python/statsmodels-omslaget. `PROC LOGISTIC` anpassar en binär
  logistisk regression med maximum likelihood.
- **Metoder för riskkomposition:** Den sammansatta inflytandepoängen i
  Avsnitt 11 normaliserar grad, log-PageRank, jurisdiktionsbredd och
  ämbetsår till `[0, 1]` och summerar med fasta vikter (30/30/20/20).
  Den sammansatta riskpoängen per entitet i Avsnitt 15 normaliserar
  begränsat antal befattningshavare, log-PageRank, CTHI-riskvikt och en
  binär flagga för sanktionerad befattningshavare och summerar med lika
  vikter på 0.25 vardera.

Hela analysen är reproducerbar från smoke-test-skriptet i den här
mappen: `./smoke.jenner`. Att köra det hela vägen mot den delade
`step-neo4j`-tjänsten (med `JENNER_NEO4J_HOST` och `JENNER_NEO4J_PASS`
satta, vilket plattformen gör åt dig i vilken workspace-podd som helst)
tar ungefär två minuter och verifierar att varje fråga och varje
PROC — inklusive de fem nya avsnitten som lagts till vid sidan av den
befintliga pipelinen — ger förväntad utdata på den verkliga grafen med
163,435 noder.